### Data extractor 

In [152]:
import json
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
from plotly.colors import hex_to_rgb
import plotly.graph_objects as go
import plotly.express as px

In [153]:
def parse_zip_experiments(zip_filepath):
    data = []
    
    with zipfile.ZipFile(zip_filepath, 'r') as archive:
        for filename in archive.namelist():
            if filename.endswith('.json'):
                with archive.open(filename) as f:
                    try:
                        exp = json.load(f)
                        
                        params = exp.get("parameters", {})
                        num_agents = params.get("numAgents")
                        num_objects = params.get("numObjects")
                        ratio_random = params.get("ratioRandom", 0) 
                        
                        results = exp.get("results", {})
                        
                        solver = results.get("solver", {})
                        solver_time = solver.get("timeUs", 0)
                        solver_score = solver.get("score", 0)
                        
                        mcts = results.get("mcts", {})
                        mcts_tries = mcts.get("tries", [])
                        
                        mcts_total_times = []
                        mcts_final_scores = []
                        
                        for t in mcts_tries:
                            steps = t.get("steps", [])
                            if steps:
                                mcts_total_times.append(sum(step.get("stepTimeUs", 0) for step in steps))
                                mcts_final_scores.append(steps[-1].get("score", 0))
                        
                        # --- NOUVEAU : Calcul des moyennes, mins et maxs ---
                        if mcts_total_times:
                            data.append({
                                "Agents": num_agents,
                                "Objets": num_objects,
                                "Ratio": ratio_random,
                                "Solveur_Temps_us": solver_time,
                                "Solveur_Score": solver_score,
                                "MCTS_Temps_Moyen_us": sum(mcts_total_times) / len(mcts_total_times),
                                "MCTS_Temps_Min_us": min(mcts_total_times),
                                "MCTS_Temps_Max_us": max(mcts_total_times),
                                "MCTS_Score_Moyen": sum(mcts_final_scores) / len(mcts_final_scores),
                                "MCTS_Score_Min": min(mcts_final_scores),
                                "MCTS_Score_Max": max(mcts_final_scores)
                            })
                        
                    except Exception as e:
                        print(f"Erreur lors de la lecture de {filename} : {e}")

    return pd.DataFrame(data)

In [154]:
def plot_comparisons(df):
    if df.empty:
        print("Aucune donnée trouvée dans le ZIP.")
        return

    # On identifie combien de configurations d'agents différentes tu as
    configurations_agents = df['Agents'].unique()
    configurations_agents.sort()

    # Pour chaque nombre d'agents, on dessine un graphique séparé
    for agents in configurations_agents:
        # On filtre pour ne garder que les données de ce nombre d'agents
        df_filtre = df[df['Agents'] == agents]
        
        # On fait la moyenne par nombre d'objets
        df_grouped = df_filtre.groupby('Objets').mean().reset_index()

        # Si on n'a qu'un seul point (un seul nombre d'objets testé pour ce nombre d'agents), on le signale
        if len(df_grouped) < 2:
            print(f"⚠️ Pas assez de variations d'objets pour tracer une courbe pour {agents} agents.")
            continue

        # Création de la figure
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        fig.suptitle(f"Analyse pour {agents} Agents", fontsize=16, fontweight='bold')

        # --- Courbe 1 : Temps d'exécution (Axe X = Objets) ---
        ax1.plot(df_grouped['Objets'], df_grouped['Solveur_Temps_us'], marker='o', label='Solveur', color='red', linewidth=2)
        ax1.plot(df_grouped['Objets'], df_grouped['MCTS_Temps_Moyen_us'], marker='s', label='MCTS (Moyenne)', color='blue', linewidth=2)
        ax1.set_title("Évolution du Temps d'exécution")
        ax1.set_xlabel("Nombre d'Objets")
        ax1.set_ylabel("Temps (microsecondes)")
        ax1.set_yscale('log') # Conserve l'échelle logarithmique pour les temps exponentiels
        ax1.legend()
        ax1.grid(True, which="both", ls="--")

        # --- Courbe 2 : Score (Axe X = Objets) ---
        ax2.plot(df_grouped['Objets'], df_grouped['Solveur_Score'], marker='o', label='Solveur', color='red', linewidth=2)
        ax2.plot(df_grouped['Objets'], df_grouped['MCTS_Score_Moyen'], marker='s', label='MCTS (Moyenne)', color='blue', linewidth=2)
        ax2.set_title("Évolution des Scores")
        ax2.set_xlabel("Nombre d'Objets")
        ax2.set_ylabel("Score")
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()
        plt.show()

In [155]:
def plot_comparisons_interactive(df):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    configurations_agents = df['Agents'].unique()
    configurations_agents.sort()

    for agents in configurations_agents:
        df_filtre = df[df['Agents'] == agents]
        
        # On fait la moyenne par nombre d'objets
        df_grouped = df_filtre.groupby('Objets').mean().reset_index()

        if len(df_grouped) < 2:
            continue

        # Création de la figure avec deux colonnes
        fig = make_subplots(
            rows=1, cols=2, 
            subplot_titles=("Temps d'exécution", "Scores Absolus")
        )

        # --- Graphique 1 : Temps ---
        fig.add_trace(go.Scatter(
            x=df_grouped['Objets'], y=df_grouped['Solveur_Temps_us'],
            mode='lines+markers', name='Solveur (Temps)', line=dict(color='red')
        ), row=1, col=1)
        
        fig.add_trace(go.Scatter(
            x=df_grouped['Objets'], y=df_grouped['MCTS_Temps_Moyen_us'],
            mode='lines+markers', name='MCTS Moyenne (Temps)', line=dict(color='blue')
        ), row=1, col=1)

        # --- Graphique 2 : Score ---
        fig.add_trace(go.Scatter(
            x=df_grouped['Objets'], y=df_grouped['Solveur_Score'],
            mode='lines+markers', name='Solveur (Score)', line=dict(color='red', dash='dot')
        ), row=1, col=2)
        
        fig.add_trace(go.Scatter(
            x=df_grouped['Objets'], y=df_grouped['MCTS_Score_Moyen'],
            mode='lines+markers', name='MCTS Moyenne (Score)', line=dict(color='blue', dash='dot')
        ), row=1, col=2)

        # --- Mise en forme ---
        fig.update_layout(
            title_text=f"Analyse globale pour {agents} Agents",
            title_font=dict(size=20),
            hovermode="x unified", # Affiche toutes les infos au même endroit au survol !
            template="plotly_white"
        )
        
        # Axe X
        fig.update_xaxes(title_text="Nombre d'Objets", row=1, col=1)
        fig.update_xaxes(title_text="Nombre d'Objets", row=1, col=2)
        
        # Axe Y (Log pour le temps)
        fig.update_yaxes(title_text="Temps (µs)", type="log", row=1, col=1)
        fig.update_yaxes(title_text="Score", row=1, col=2)

        fig.show()

In [156]:
def plot_impact_ratio(df):
    if df.empty:
        print("Aucune donnée trouvée dans le ZIP.")
        return

    # On sépare les graphiques par nombre d'agents pour que ce soit lisible
    configurations_agents = df['Agents'].unique()
    configurations_agents.sort()

    for agents in configurations_agents:
        df_filtre = df[df['Agents'] == agents]
        
        # On groupe par Ratio pour voir son impact global (moyenne sur tous les objets et seeds)
        df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

        if len(df_grouped) < 2:
            print(f"⚠️ Pas assez de variations de 'ratioRandom' pour tracer une courbe pour {agents} agents.")
            continue

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        fig.suptitle(f"Impact du Ratio Random ({agents} Agents)", fontsize=16, fontweight='bold')

        # --- Courbe 1 : Temps d'exécution (Axe X = Ratio) ---
        # Le solveur est tracé pour vérifier si le ratio impacte la génération du problème
        ax1.plot(df_grouped['Ratio'], df_grouped['Solveur_Temps_us'], marker='o', label='Solveur', color='red', linewidth=2)
        ax1.plot(df_grouped['Ratio'], df_grouped['MCTS_Temps_Moyen_us'], marker='s', label='MCTS (Moyenne)', color='blue', linewidth=2)
        ax1.set_title("Évolution du Temps selon le Ratio")
        ax1.set_xlabel("Ratio Random")
        ax1.set_ylabel("Temps (microsecondes)")
        ax1.set_yscale('log')
        ax1.legend()
        ax1.grid(True, which="both", ls="--")

        # --- Courbe 2 : Score (Axe X = Ratio) ---
        ax2.plot(df_grouped['Ratio'], df_grouped['Solveur_Score'], marker='o', label='Solveur (Optimum)', color='red', linewidth=2)
        ax2.plot(df_grouped['Ratio'], df_grouped['MCTS_Score_Moyen'], marker='s', label='MCTS (Moyenne)', color='blue', linewidth=2)
        ax2.set_title("Évolution des Scores selon le Ratio")
        ax2.set_xlabel("Ratio Random")
        ax2.set_ylabel("Score")
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()
        plt.show()

In [157]:
def plot_impact_ratio_interactive(df):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    configurations_agents = df['Agents'].unique()
    configurations_agents.sort()

    for agents in configurations_agents:
        df_filtre = df[df['Agents'] == agents]
        
        # On groupe par Ratio
        df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

        if len(df_grouped) < 2:
            continue

        # Création de la figure avec deux colonnes
        fig = make_subplots(
            rows=1, cols=2, 
            subplot_titles=("Temps selon le Ratio", "Score selon le Ratio")
        )

        # --- Graphique 1 : Temps ---
        fig.add_trace(go.Scatter(
            x=df_grouped['Ratio'], y=df_grouped['Solveur_Temps_us'],
            mode='lines+markers', name='Solveur (Temps)', line=dict(color='red')
        ), row=1, col=1)
        
        fig.add_trace(go.Scatter(
            x=df_grouped['Ratio'], y=df_grouped['MCTS_Temps_Moyen_us'],
            mode='lines+markers', name='MCTS (Temps)', line=dict(color='blue')
        ), row=1, col=1)

        # --- Graphique 2 : Score ---
        fig.add_trace(go.Scatter(
            x=df_grouped['Ratio'], y=df_grouped['Solveur_Score'],
            mode='lines+markers', name='Solveur (Score)', line=dict(color='red', dash='dot')
        ), row=1, col=2)
        
        fig.add_trace(go.Scatter(
            x=df_grouped['Ratio'], y=df_grouped['MCTS_Score_Moyen'],
            mode='lines+markers', name='MCTS (Score)', line=dict(color='blue', dash='dot')
        ), row=1, col=2)

        # --- Mise en forme ---
        fig.update_layout(
            title_text=f"Impact du Ratio Random pour {agents} Agents",
            title_font=dict(size=20),
            hovermode="x unified", # Le tooltip global très lisible
            template="plotly_white"
        )
        
        # Axe X
        fig.update_xaxes(title_text="Ratio Random", row=1, col=1)
        fig.update_xaxes(title_text="Ratio Random", row=1, col=2)
        
        # Axe Y (Log pour le temps)
        fig.update_yaxes(title_text="Temps (µs)", type="log", row=1, col=1)
        fig.update_yaxes(title_text="Score", row=1, col=2)

        fig.show()

In [158]:
def plot_ratio_impact_on_score(df, combine_agents=True):
    if df.empty:
        print("Aucune donnée trouvée dans le ZIP.")
        return

    # On copie le DataFrame et on s'assure de ne pas diviser par zéro
    df_score = df[df['Solveur_Score'] > 0].copy()
    
    # Calcul du pourcentage d'optimalité
    df_score['Optimalite_MCTS_%'] = (df_score['MCTS_Score_Moyen'] / df_score['Solveur_Score']) * 100

    configurations_agents = df_score['Agents'].unique()
    configurations_agents.sort()

    # --- CAS 1 : Tout sur le même graphique ---
    if combine_agents:
        plt.figure(figsize=(8, 5)) 

        for agents in configurations_agents:
            df_filtre = df_score[df_score['Agents'] == agents]
            df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

            if len(df_grouped) < 2:
                print(f"⚠️ Pas assez de variations de 'ratioRandom' pour {agents} agents.")
                continue
            
            plt.plot(df_grouped['Ratio'], df_grouped['Optimalite_MCTS_%'], 
                     marker='o', linewidth=2, label=f'MCTS ({agents} Agents)')

        plt.axhline(y=100, color='red', linestyle='--', linewidth=2, label='Optimum (Solveur = 100%)')
        plt.title("Impact du Ratio sur la qualité du score (Toutes configurations)", fontsize=14, fontweight='bold')
        plt.xlabel("Ratio Random")
        plt.ylabel("Qualité de la solution (%)")
        plt.ylim(0, 105) 
        plt.legend()
        plt.grid(True, which="both", ls="--")
        plt.tight_layout()
        plt.show()

    # --- CAS 2 : Un graphique séparé par nombre d'agents ---
    else:
        for agents in configurations_agents:
            df_filtre = df_score[df_score['Agents'] == agents]
            df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

            if len(df_grouped) < 2:
                print(f"⚠️ Pas assez de variations de 'ratioRandom' pour {agents} agents.")
                continue

            plt.figure(figsize=(7, 4))
            
            plt.plot(df_grouped['Ratio'], df_grouped['Optimalite_MCTS_%'], 
                     marker='o', color='purple', linewidth=2, label='Score MCTS')
            plt.axhline(y=100, color='red', linestyle='--', linewidth=2, label='Optimum (Solveur = 100%)')
            
            plt.title(f"Impact du Ratio sur la qualité du score ({agents} Agents)", fontsize=14, fontweight='bold')
            plt.xlabel("Ratio Random")
            plt.ylabel("Qualité de la solution (%)")
            plt.ylim(0, 105) 
            plt.legend()
            plt.grid(True, which="both", ls="--")
            plt.tight_layout()
            plt.show()

In [159]:
def plot_ratio_impact_on_score_interactive(df, combine_agents=True):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    df_score = df[df['Solveur_Score'] > 0].copy()
    df_score['Optimalite_MCTS_%'] = (df_score['MCTS_Score_Moyen'] / df_score['Solveur_Score']) * 100
    df_score['Optimalite_MCTS_%'] = df_score['Optimalite_MCTS_%'].clip(upper=100)

    # --- CHANGEMENT 1 : Calcul du Min, Max et de la Moyenne ---
    df_grouped = df_score.groupby(['Agents', 'Ratio']).agg(
        Optimalite_Moyenne=('Optimalite_MCTS_%', 'mean'),
        Optimalite_Min=('Optimalite_MCTS_%', 'min'),
        Optimalite_Max=('Optimalite_MCTS_%', 'max'),
        Score_MCTS_Moyen=('MCTS_Score_Moyen', 'mean'),
        Score_Solveur=('Solveur_Score', 'mean')
    ).reset_index()
    
    df_grouped['Configuration'] = df_grouped['Agents'].astype(str) + " Agents"

    if combine_agents:
        y_min = max(0, df_grouped['Optimalite_Min'].min() )
        y_max = 100
        
        fig = px.line(
            df_grouped,
            x="Ratio",
            y="Optimalite_Moyenne", # On pointe vers la nouvelle colonne
            color="Configuration",
            markers=True,
            title="Impact du Ratio sur la qualité du score (Toutes configurations)",
            labels={
                "Ratio": "Ratio Random",
                "Optimalite_Moyenne": "Qualité de la solution (%)"
            },
            # --- CHANGEMENT 2 : Ajout du Min et Max dans l'infobulle ---
            hover_data={
                "Configuration": False, 
                "Ratio": ':.2f',
                "Optimalite_Moyenne": ':.2f', 
                "Optimalite_Min": ':.2f',  # <- Ajout
                "Optimalite_Max": ':.2f',  # <- Ajout
                "Score_MCTS_Moyen": ':.4f',
                "Score_Solveur": ':.4f'
            }
        )
        
        fig.add_hline(y=100, line_dash="dash", line_color="red", annotation_text="Optimum (100%)")
        fig.update_layout(yaxis=dict(range=[y_min, y_max]), hovermode="x unified", template="plotly_white")
        fig.show()

    else:
        configurations_agents = df_grouped['Agents'].unique()
        configurations_agents.sort()
        
        for agents in configurations_agents:
            df_filtre = df_grouped[df_grouped['Agents'] == agents]
            
            if len(df_filtre) < 2:
                continue

            y_min_local = max(0, df_filtre['Optimalite_Min'].min() - 2)
            y_max_local = 102
            
            fig = px.line(
                df_filtre,
                x="Ratio",
                y="Optimalite_Moyenne", # On pointe vers la nouvelle colonne
                markers=True,
                title=f"Impact du Ratio sur la qualité du score ({agents} Agents)",
                labels={
                    "Ratio": "Ratio Random",
                    "Optimalite_Moyenne": "Qualité de la solution (%)"
                },
                color_discrete_sequence=['purple'],
                # --- CHANGEMENT 2 : Ajout du Min et Max dans l'infobulle ---
                hover_data={
                    "Ratio": ':.2f',
                    "Optimalite_Moyenne": ':.2f', 
                    "Optimalite_Min": ':.2f',  # <- Ajout
                    "Optimalite_Max": ':.2f',  # <- Ajout
                    "Score_MCTS_Moyen": ':.4f',
                    "Score_Solveur": ':.4f'
                }
            )
            
            fig.add_hline(y=100, line_dash="dash", line_color="red", annotation_text="Optimum (100%)")
            fig.update_layout(yaxis=dict(range=[y_min_local, y_max_local]), hovermode="x unified", template="plotly_white")
            fig.show()

In [160]:
def plot_erreur_normalisee(df, combine_agents=True):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    # On copie le DataFrame et on s'assure de ne pas diviser par zéro
    df_safe = df[df['Solveur_Score'] > 0].copy()

    # Calcul de l'erreur normalisée globale en pourcentage pour chaque ligne
    df_safe['Erreur_%'] = ((df_safe['Solveur_Score'] - df_safe['MCTS_Score_Moyen']) / df_safe['Solveur_Score']) * 100

    configurations_agents = df_safe['Agents'].unique()
    configurations_agents.sort()

    # --- CAS 1 : Toutes les courbes sur le même graphique ---
    if combine_agents:
        plt.figure(figsize=(10, 6))

        for agents in configurations_agents:
            df_filtre = df_safe[df_safe['Agents'] == agents]
            df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

            if len(df_grouped) < 2:
                continue
                
            plt.plot(df_grouped['Ratio'], df_grouped['Erreur_%'], 
                     marker='o', linewidth=2, label=f'MCTS ({agents} Agents)')

        plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Optimum (Erreur = 0%)')

        plt.title("Erreur Normalisée globale du MCTS selon le Ratio", fontsize=14, fontweight='bold')
        plt.xlabel("Ratio Random")
        plt.ylabel("Erreur par rapport au Solveur (%)")
        
        erreur_max = df_safe['Erreur_%'].max()
        plt.ylim(-1, erreur_max + 2 if pd.notna(erreur_max) else 10)
        
        plt.grid(True, which="both", ls="--")
        plt.legend()
        plt.tight_layout()
        plt.show()

    # --- CAS 2 : Un graphique séparé par configuration d'agents ---
    else:
        for agents in configurations_agents:
            df_filtre = df_safe[df_safe['Agents'] == agents]
            df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

            if len(df_grouped) < 2:
                continue
            
            plt.figure(figsize=(7, 4))
            
            # Utilisation d'une couleur neutre si c'est affiché seul
            plt.plot(df_grouped['Ratio'], df_grouped['Erreur_%'], 
                     marker='o', color='purple', linewidth=2, label='Erreur MCTS')

            plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Optimum (Erreur = 0%)')

            plt.title(f"Erreur Normalisée du MCTS selon le Ratio ({agents} Agents)", fontsize=14, fontweight='bold')
            plt.xlabel("Ratio Random")
            plt.ylabel("Erreur par rapport au Solveur (%)")
            
            # Ajustement dynamique du zoom pour ce graphique spécifique
            erreur_max = df_filtre['Erreur_%'].max()
            plt.ylim(-1, erreur_max + 2 if pd.notna(erreur_max) else 10)
            
            plt.grid(True, which="both", ls="--")
            plt.legend()
            plt.tight_layout()
            plt.show()

In [161]:
def plot_erreur_normalisee_interactive(df, combine_agents=True):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    # Préparation des données
    df_safe = df[df['Solveur_Score'] > 0].copy()
    df_safe['Erreur_%'] = ((df_safe['Solveur_Score'] - df_safe['MCTS_Score_Moyen']) / df_safe['Solveur_Score']) * 100

    # On groupe par Agents et Ratio pour faire les moyennes
    df_grouped = df_safe.groupby(['Agents', 'Ratio']).mean().reset_index()
    
    # On crée une colonne texte pour que Plotly gère bien les catégories/couleurs
    df_grouped['Configuration'] = df_grouped['Agents'].astype(str) + " Agents"

    if combine_agents:
        # --- Création du graphique interactif (Toutes les courbes) ---
        fig = px.line(
            df_grouped,
            x="Ratio",
            y="Erreur_%",
            color="Configuration",
            markers=True,
            title="Erreur Normalisée globale du MCTS selon le Ratio",
            labels={
                "Ratio": "Ratio Random",
                "Erreur_%": "Erreur par rapport au Solveur (%)"
            },
            # Configuration de la bulle d'info au survol (hover)
            hover_data={
                "Configuration": False, # On le cache de la liste car c'est le titre de la bulle
                "Ratio": ':.2f',
                "Erreur_%": ':.2f', 
                "MCTS_Score_Moyen": ':.2f',
                "Solveur_Score": ':.2f'
            }
        )
        
        # Ajout de la ligne rouge de l'optimum
        fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Optimum (0%)")
        
        # Forcer l'axe Y à descendre un peu sous le 0
        fig.update_layout(yaxis=dict(range=[-1, df_grouped['Erreur_%'].max() + 2]))
        
        # Ouvre le graphique dans ton navigateur web
        fig.show()

    else:
        # --- Création de graphiques séparés ---
        configurations_agents = df_grouped['Agents'].unique()
        configurations_agents.sort()
        
        for agents in configurations_agents:
            df_filtre = df_grouped[df_grouped['Agents'] == agents]
            
            fig = px.line(
                df_filtre,
                x="Ratio",
                y="Erreur_%",
                markers=True,
                title=f"Erreur Normalisée du MCTS selon le Ratio ({agents} Agents)",
                labels={
                    "Ratio": "Ratio Random",
                    "Erreur_%": "Erreur par rapport au Solveur (%)"
                },
                color_discrete_sequence=['purple'],
                hover_data={
                    "Ratio": ':.2f',
                    "Erreur_%": ':.2f', 
                    "MCTS_Score_Moyen": ':.2f',
                    "Solveur_Score": ':.2f'
                }
            )
            
            fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Optimum (0%)")
            fig.update_layout(yaxis=dict(range=[-1, df_filtre['Erreur_%'].max() + 2]))
            fig.show()

In [162]:
# Exécution
nom_du_fichier_zip = "../" + "results/experiments_04-06-2026_08-58-59.zip" 
df = parse_zip_experiments(nom_du_fichier_zip)

In [163]:
#plot_comparisons(df)

In [164]:
#plot_impact_ratio(df)

In [165]:
#plot_ratio_impact_on_score(df)

In [166]:
#plot_erreur_normalisee(df)

In [168]:
plot_comparisons_interactive(df)
plot_impact_ratio_interactive(df)
plot_ratio_impact_on_score_interactive(df)
plot_erreur_normalisee_interactive(df)
